# Exercise: Regularise the Forest Cover Classifier

The Covertype data describes forest areas by their terrain, and the task is to predict the tree species that grows there. A baseline network with two hidden layers reaches a validation accuracy of about 0.82 after 30 epochs, and its learning curves are still improving. In this exercise you use what you learnt today to improve it: you train it longer, make it larger, diagnose its learning curves and regularise it, comparing every version on the validation set. At the end you evaluate your chosen model once on the test set.

The tasks describe what to build and what to report; the code is yours to write. Notebooks 01 and 02 contain the training settings and the regularisation tools you need; the data preparation and the accuracy metric are those you used for multiclass classification before. A larger network trains for one to two minutes, so plan your runs before you start them.

## Learning Objectives

By the end of this exercise, you should be able to:

- Train a Keras classifier on real data and diagnose its learning curves for loss and accuracy.
- Apply L2 regularisation, dropout and early stopping to a larger network and compare the versions on a validation set.
- Choose the value to monitor in `EarlyStopping` and `ModelCheckpoint` that matches the goal of the model.
- Evaluate the chosen model once on the test set and compare it with the baseline network.

## Setup

In [ ]:
from pathlib import Path

import keras
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.datasets import fetch_covtype
from sklearn.model_selection import train_test_split

# One plot style for the whole notebook
plt.rcParams.update(
    {"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3}
)

## Step 1: Load the Data

The data is a 60,000-row sample of the [Covertype dataset](https://archive.ics.uci.edu/dataset/31/covertype), loaded with [`fetch_covtype`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_covtype.html): 54 features (10 terrain measurements and 44 one-hot encoded wilderness and soil columns) and seven tree species as the target. The cell below loads it.

In [ ]:
covtype = fetch_covtype(as_frame=True)
forest = covtype.frame.sample(60_000, random_state=42)

class_names = [
    "Spruce/Fir", "Lodgepole Pine", "Ponderosa Pine", "Cottonwood/Willow",
    "Aspen", "Douglas-fir", "Krummholz",
]
X = forest.drop(columns="Cover_Type")
y = forest["Cover_Type"] - 1  # codes 1-7 become labels 0-6
print("Rows and columns:", X.shape)

## Step 2: Split the Data and Prepare the Normalisation

Split the data as follows, so your numbers are comparable with the solution:

- First keep 20 % of the rows as the test set, then 20 % of the rest as the validation set.
- Use `stratify` and `random_state=42` in both splits.
- Print the shapes of `X_train`, `X_valid` and `X_test`.

Then create a [`Normalization`](https://keras.io/api/layers/preprocessing_layers/numerical/normalization/) layer and adapt it to the training features. Every model in this exercise reuses this layer.

In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=0.2, stratify=y_train_full, random_state=42
)
print("Train:", X_train.shape, " Validation:", X_valid.shape, " Test:", X_test.shape)

norm_layer = keras.layers.Normalization()
norm_layer.adapt(X_train.to_numpy())  # learn the mean and variance of each feature from the training set

## Step 3: Plan How You Train

You will train about ten versions of the same classifier. How you organise the code is up to you:

- **Write helper functions** (optional), similar to those in notebook 02. For example:
  1. `build_model(units, l2_strength=0.0, dropout_rate=0.0)` returns a new `Sequential` model: the input for 54 features, the normalisation layer, one ReLU `Dense` layer per entry of `units` (with L2 regularisation and a `Dropout` layer after each when the arguments are above 0), and an output layer for the seven classes.
  2. `train_model(model, epochs, callbacks=None)` compiles and trains the model and returns the history as a DataFrame.
  3. `plot_curves(logs, title)` draws the training and validation loss next to the training and validation accuracy.
- **Write each model out in full**, and copy and change the cells from one step to the next.

Whichever way you choose, keep these settings the same for every version, so the comparison is fair and your numbers are close to the solution's:

- Call `keras.utils.clear_session()` and `keras.utils.set_random_seed(42)` before you build each model.
- Choose the output layer and the loss for a multiclass problem with integer labels (notebook 01, section 2), and use them for every version.
- Compile with Adam (learning rate 0.001) and the accuracy metric.
- Train with `batch_size=128` and the validation set as `validation_data`.
- Plot the loss and the accuracy curves of every run, for both the training and the validation set.

Also create an empty dictionary `results`. After every version, store its validation accuracy there under a descriptive name, so you can compare all versions in step 9.

The cell below is for your helper functions, if you write them; you can also leave it empty.

In [ ]:
def build_model(units, l2_strength=0.0, dropout_rate=0.0):
    """Return a new classifier with ReLU hidden layers, optional L2 regularisation and dropout."""
    keras.utils.clear_session()  # start from a clean Keras state
    keras.utils.set_random_seed(42)  # the same starting weights for the same settings
    regularizer = keras.regularizers.L2(l2_strength) if l2_strength > 0 else None
    layers = [keras.Input(shape=(X_train.shape[1],)), norm_layer]
    for n_neurons in units:
        layers.append(keras.layers.Dense(units=n_neurons, activation="relu", kernel_regularizer=regularizer))
        if dropout_rate > 0:
            layers.append(keras.layers.Dropout(dropout_rate))
    layers.append(keras.layers.Dense(units=7, activation="softmax"))  # one probability per species
    return keras.Sequential(layers)


def train_model(model, epochs, callbacks=None):
    """Compile and train a classifier; return its learning curves as a DataFrame."""
    model.compile(
        loss="sparse_categorical_crossentropy",
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        metrics=["accuracy"],
    )
    history = model.fit(
        X_train, y_train, epochs=epochs, batch_size=128, validation_data=(X_valid, y_valid),
        callbacks=callbacks, verbose=0,
    )
    return pd.DataFrame(history.history)


def plot_curves(logs, title):
    """Plot the training and validation loss and accuracy of one run."""
    fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(12, 4))
    logs[["loss", "val_loss"]].plot(ax=ax_loss, xlabel="Epoch", title=f"{title}: loss")
    logs[["accuracy", "val_accuracy"]].plot(ax=ax_acc, xlabel="Epoch", title=f"{title}: accuracy")
    plt.show()


results = {}

## Step 4: The Baseline Network, Trained Longer

Build the baseline network: two hidden layers of 128 and 64 neurons, no regularisation.

1. Train it for 30 epochs and store its validation accuracy. It should be close to 0.82.
2. Build it again and train it for 100 epochs. Plot the learning curves and store the validation accuracy.

Which panel of the learning-curves figure in notebook 02 describes the 100-epoch run best? How much did the extra 70 epochs improve the validation accuracy?

In [ ]:
baseline_model = build_model(units=(128, 64))
train_model(baseline_model, epochs=30)
results["Baseline network, 30 epochs"] = baseline_model.evaluate(X_valid, y_valid, verbose=0)[1]

long_model = build_model(units=(128, 64))
long_logs = train_model(long_model, epochs=100)
plot_curves(long_logs, "Baseline network, 100 epochs")
results["Baseline network, 100 epochs"] = long_model.evaluate(X_valid, y_valid, verbose=0)[1]
pd.Series(results).round(3)

**Result.** After 30 epochs the baseline network reaches a validation accuracy of 0.819. After 100 epochs it reaches 0.842. The learning curves show where the extra epochs went: the validation accuracy rises quickly until about epoch 40 and only slowly afterwards, while the training accuracy keeps rising to 0.90. The validation loss stops falling at about 0.41 after epoch 50, while the training loss keeps falling. The run is between panel A (a good fit) and the start of panel C (overfitting): the extra 70 epochs added 2.3 percentage points of validation accuracy, and the growing gap between the curves shows that further epochs would mainly improve the fit to the training set.

## Step 5: A Larger Network

The validation accuracy is still rising slowly, so give the network more capacity: three hidden layers of 512, 256 and 128 neurons. Train it for 100 epochs, plot the learning curves and store the validation accuracy.

Look at both plots:

- At which epoch is the validation loss lowest, and what does it do afterwards?
- What does the validation accuracy do over the same epochs?
- Which panel of the learning-curves figure describes the loss curves? How can the validation loss rise while the validation accuracy also rises? Hint: accuracy only counts whether the class with the highest probability is correct, while cross-entropy also measures how much probability the model gives to the correct class.

In [ ]:
large_units = (512, 256, 128)
large_model = build_model(units=large_units)
large_logs = train_model(large_model, epochs=100)
plot_curves(large_logs, "Large network")
results["Large network"] = large_model.evaluate(X_valid, y_valid, verbose=0)[1]

best_loss_epoch = large_logs["val_loss"].idxmin() + 1  # epochs are counted from 1
print(f"Lowest validation loss {large_logs['val_loss'].min():.3f} at epoch {best_loss_epoch}; "
      f"after 100 epochs {large_logs['val_loss'].iloc[-1]:.3f}")
print(f"Validation accuracy at epoch {best_loss_epoch}: {large_logs['val_accuracy'].iloc[best_loss_epoch - 1]:.3f}; "
      f"after 100 epochs {large_logs['val_accuracy'].iloc[-1]:.3f}")

**Result.** The larger network reaches a validation accuracy of 0.873, much better than the 0.842 of the baseline network. Its loss curves, however, look like panel C: the validation loss is lowest at epoch 16 (0.407) and then rises steadily to 0.685, while the training loss falls to below 0.1. At epoch 16 the validation accuracy is 0.839; it keeps rising to 0.873 while the validation loss rises.

Both curves are correct, and they measure different things. Accuracy only counts whether the class with the highest probability is the correct one. Cross-entropy also measures how much probability the model gives to the correct class, and it grows quickly when the model is confidently wrong. As the large network memorises the training set, it becomes more confident in all its predictions: it gets more validation rows right, which raises the accuracy, but on the rows it still gets wrong it now gives the correct class a very low probability, which raises the loss. The network is overfitting in its probabilities, not yet in its decisions.

## Step 6: L2 Regularisation

Train the large network for 100 epochs with L2 regularisation in every hidden layer. Try two strengths, $\lambda = 0.0001$ and $\lambda = 0.001$. Plot the learning curves of each run and store the validation accuracies.

Compare the curves with those of step 5: does L2 reduce the gap between training and validation accuracy? Does it improve the validation accuracy? Keras adds the L2 penalty to the reported loss, so compare the accuracy curves, not the loss values, with step 5.

In [ ]:
for l2_strength in [1e-4, 1e-3]:
    l2_model = build_model(units=large_units, l2_strength=l2_strength)
    l2_logs = train_model(l2_model, epochs=100)
    plot_curves(l2_logs, f"Large network with L2 ({l2_strength})")
    results[f"Large network with L2 ({l2_strength})"] = l2_model.evaluate(X_valid, y_valid, verbose=0)[1]
pd.Series(results).round(3)

**Result.** L2 regularisation does not help here. With $\lambda = 0.0001$ the validation accuracy is 0.860 and with $\lambda = 0.001$ it is 0.839, both below the 0.873 of the large network without regularisation. With $\lambda = 0.0001$ the accuracy curves still separate, from about 0.95 on the training set to about 0.86 on the validation set, and the validation loss rises again after epoch 25. (Its loss values include the L2 penalty, so they cannot be compared directly with those of step 5.) The penalty slows the overfitting of the probabilities, but it also limits how well the network fits the many small patterns in this data. The stronger penalty limits it more and lowers the accuracy further.

## Step 7: Dropout

Train the large network for 100 epochs with dropout after every hidden layer. Try two rates, 0.2 and 0.4. Plot the learning curves of each run and store the validation accuracies.

Which rate works better? Are the curves still improving at epoch 100?

In [ ]:
for dropout_rate in [0.2, 0.4]:
    dropout_model = build_model(units=large_units, dropout_rate=dropout_rate)
    dropout_logs = train_model(dropout_model, epochs=100)
    plot_curves(dropout_logs, f"Large network with dropout ({dropout_rate})")
    results[f"Large network with dropout ({dropout_rate})"] = dropout_model.evaluate(X_valid, y_valid, verbose=0)[1]
pd.Series(results).round(3)

**Result.** Dropout with a rate of 0.2 gives the best validation accuracy so far, 0.882; a rate of 0.4 gives 0.860, so too much dropout holds the network back. With a rate of 0.2 the learning curves look very different from those of step 5: the training and validation loss fall together until about epoch 20, and afterwards the validation loss stays flat at about 0.32 instead of rising to 0.69. The validation accuracy is still rising slowly at epoch 100, so this model can benefit from more epochs.

## Step 8: Early Stopping: What Should It Watch?

Train the large network without regularisation twice with an [`EarlyStopping`](https://keras.io/api/callbacks/early_stopping/) callback (at most 100 epochs, `patience=10`, `restore_best_weights=True`):

1. once with `monitor="val_loss"`,
2. once with `monitor="val_accuracy"`.

For each run, print the number of epochs it trained and store the validation accuracy of the restored model.

Which of the two monitors fits the goal of this model, which is to predict the correct species as often as possible? Use your answer to step 5.

In [ ]:
for monitor in ["val_loss", "val_accuracy"]:
    early_stopping = keras.callbacks.EarlyStopping(monitor=monitor, patience=10, restore_best_weights=True)
    early_model = build_model(units=large_units)
    early_logs = train_model(early_model, epochs=100, callbacks=[early_stopping])
    name = f"Large network, early stopping on {monitor}"
    results[name] = early_model.evaluate(X_valid, y_valid, verbose=0)[1]
    print(f"{name}: trained {len(early_logs)} epochs, validation accuracy {results[name]:.3f}")

**Result.** With `monitor="val_loss"` training stops after 26 epochs and restores the weights of epoch 16, where the validation loss was lowest. Their validation accuracy is 0.839, lower than the 0.873 that the same network reached after 100 epochs. With `monitor="val_accuracy"` training runs for 54 epochs and the restored model reaches 0.865. That is still below the 0.873 of 100 epochs without early stopping: the validation accuracy rises slowly and with small ups and downs, so ten epochs without a new best happen even while the model is still improving. A patience of 10 is too short for such a curve, which is why the final model in step 9 uses `patience=20`.

The goal of this model is to predict the correct species as often as possible, so `val_accuracy` is the value to monitor. Step 5 showed that the validation loss of this network rises for reasons that do not reduce its accuracy, so stopping on the loss ends training too early. Monitoring the loss is the better choice when the probabilities themselves matter, for example when a decision depends on how confident the model is.

## Step 9: Your Final Model

Combine what worked best in steps 6 to 8 into one final model:

- Choose the regularisation from your results.
- Train for up to 200 epochs with early stopping (`patience=20`), so the model can train for longer while it improves.
- Save the best epoch with a [`ModelCheckpoint`](https://keras.io/api/callbacks/model_checkpoint/) to `models/forest_cover_best.keras`, monitoring the same value as the early stopping.
- Load the saved model with `keras.saving.load_model()`, store its validation accuracy and plot the learning curves.

Then show all your versions in one table, sorted by validation accuracy.

In [ ]:
Path("models").mkdir(exist_ok=True)
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=20, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint("models/forest_cover_best.keras", monitor="val_accuracy", save_best_only=True),
]
final_model = build_model(units=large_units, dropout_rate=0.2)  # dropout 0.2 gave the best validation accuracy
final_logs = train_model(final_model, epochs=200, callbacks=callbacks)
print(f"Trained {len(final_logs)} epochs; best validation accuracy at epoch {final_logs['val_accuracy'].idxmax() + 1}")
plot_curves(final_logs, "Final model")

best_model = keras.saving.load_model("models/forest_cover_best.keras")
results["Final model: dropout 0.2, up to 200 epochs"] = best_model.evaluate(X_valid, y_valid, verbose=0)[1]
pd.Series(results, name="Validation accuracy").sort_values(ascending=False).round(3).to_frame()

**Result.** The final model combines the large network with dropout at a rate of 0.2, the best version of steps 6 to 8, with early stopping and a checkpoint that both monitor the validation accuracy. It trained for 172 epochs: the validation accuracy was highest at epoch 152 and did not improve in the 20 epochs that followed. The model loaded from the checkpoint reaches a validation accuracy of 0.891, the best in the table. Its validation loss stays flat at about 0.32 from epoch 60 onwards, so the longer training improved the decisions without making the probabilities overconfident.

## Step 10: Evaluate Once on the Test Set

Evaluate your final model on the test set. The choice is already made, so also report the test accuracy of the 30-epoch baseline network from step 4 for reference only. For comparison, logistic regression reaches a test accuracy of 0.724 on this split.

Then calculate the accuracy of the final model within each of the seven species, the share of test rows of that species it predicts correctly (the **recall** of the class). For the baseline network, the three rare species are the weakest: Cottonwood/Willow 0.38, Aspen 0.47 and Douglas-fir 0.51. Have regularisation and the larger network helped them?

In [ ]:
test_accuracy = {
    "Baseline network, 30 epochs": baseline_model.evaluate(X_test, y_test, verbose=0)[1],
    "Final model": best_model.evaluate(X_test, y_test, verbose=0)[1],
}
print(pd.Series(test_accuracy, name="Test accuracy").round(3).to_frame())

y_pred = best_model.predict(X_test, verbose=0).argmax(axis=1)
per_class = pd.DataFrame({"Actual": y_test.to_numpy(), "Correct": y_pred == y_test.to_numpy()})
per_class = per_class.groupby("Actual")["Correct"].agg(["mean", "size"])
per_class.index = class_names
per_class.columns = ["Accuracy", "Test rows"]
per_class.sort_values("Accuracy").round(2)

**Result.** On the test set the final model reaches an accuracy of 0.891, compared with 0.819 for the baseline network and 0.724 for logistic regression. The baseline network misclassifies 18.1 % of the test rows and the final model 10.9 %, so the larger, regularised network removes about 40 % of the baseline network's errors. Part of this gain comes from training longer: the same baseline reached 0.842 on the validation set after 100 epochs (step 4), so capacity and dropout account for the rest. The test accuracy equals the validation accuracy, so the choices made on the validation set hold for new data.

The recall of the rare species has improved most: Cottonwood/Willow from 0.38 to 0.68, Aspen from 0.47 to 0.69 and Douglas-fir from 0.51 to 0.76. They are still the weakest classes, because the training set has few examples of them.

## Reflection

Answer these questions in the Markdown cell below.

1. In step 5 the validation loss and the validation accuracy pointed in different directions. Which one would you use to choose a model for this problem, and why?
2. Which regularisation technique helped most here, and what did its learning curves look like compared with the large network without regularisation?
3. Every decision in this exercise was made on the validation set. Why does that make the validation accuracy of your final model slightly optimistic, and how does the test set solve this?
4. Your final model is larger and trains longer than the baseline network. Name one situation in which you would still choose the smaller network.

**Answers.**

1. The validation accuracy, because the model is used to pick one species per area, and accuracy measures exactly how often that choice is correct. The validation loss is the better guide when the predicted probabilities are used directly, for example to flag areas where the model is uncertain.
2. Dropout at a rate of 0.2. Its validation loss stayed flat at about 0.32 instead of rising to 0.69, and the gap between the training and validation curves stayed small, while the validation accuracy kept rising. L2 slowed the rise of the validation loss but lowered the accuracy.
3. About ten versions were compared on the same 9,600 validation rows, and the best one was kept. Some of its advantage can come from suiting these particular rows, so its validation accuracy slightly overestimates its accuracy on new data. The test set played no part in any decision, so its accuracy is an unbiased estimate. Here the two agree (0.891 on both).
4. When predictions must be fast or the model must run on a small device, when training time is limited, or when the smaller model is accurate enough for the task: the baseline network has about 16,000 parameters, the final model about 193,000.

## References

- Géron, A. (2022). *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow*, 3rd edition, chapter 11: Avoiding Overfitting Through Regularization.
- Keras API: [`Dropout`](https://keras.io/api/layers/regularization_layers/dropout/), [layer weight regularisers](https://keras.io/api/layers/regularizers/), [`EarlyStopping`](https://keras.io/api/callbacks/early_stopping/) and [`ModelCheckpoint`](https://keras.io/api/callbacks/model_checkpoint/).
- Guo, C., Pleiss, G., Sun, Y. and Weinberger, K. Q. (2017). [On Calibration of Modern Neural Networks](https://proceedings.mlr.press/v70/guo17a.html). *Proceedings of the 34th International Conference on Machine Learning*: why large networks become overconfident while their accuracy still improves.
- Blackard, J. (1998). [Covertype dataset](https://archive.ics.uci.edu/dataset/31/covertype). UCI Machine Learning Repository.